# 04 Evaluation

This notebook runs the current baseline suite on real held-out trajectories. It is intended to catch incorrect modeling assumptions quickly by showing both summary metrics and concrete failure cases from the real data.


## Goals

1. Build a small real-data holdout split.
2. Compare persistence, linear extrapolation, and the local empirical variants.
3. Inspect horizon curves, support correlations, and failure examples on the held-out real embryos.


In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "dev" / "particle_prediction" / "data" / "loading.py").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dev.particle_prediction.data.loading import DEFAULT_BUILD_DIR, FILE_PREFIX, load_trajectories
from dev.particle_prediction.data.smoothing import smooth_trajectory, smooth_trajectories
from dev.particle_prediction.data.resampling import compute_cumulative_arc_length, resample_smoothed_trajectory, resample_smoothed_trajectories
from dev.particle_prediction.data.transition_windows import build_transition_windows
from dev.particle_prediction.data.transition_bank import build_transition_bank
from dev.particle_prediction.data.dataset import build_prediction_tasks, build_query_from_resampled_trajectory
from dev.particle_prediction.eval.evaluate import comparison_table, run_evaluation_suite
from dev.particle_prediction.models.local_transition_pf import LocalTransitionPredictor
from dev.particle_prediction.models.matching import MatchingConfig, compare_matching_modes
from dev.particle_prediction.viz.smoothing import (
    plot_latent_trajectory_before_after_smoothing,
    plot_raw_vs_smoothed_timeseries,
    plot_sg_parameter_sweep,
)
from dev.particle_prediction.viz.transition_bank import (
    plot_arc_length_vs_time,
    plot_bank_state_density,
    plot_history_segments_example,
    plot_increment_norm_distribution,
    plot_resampled_points_on_trajectory,
    plot_transition_windows_for_embryo,
)
from dev.particle_prediction.viz.matching import (
    compare_default_vs_fast_matching,
    plot_history_offset_heatmap,
    plot_history_reranking,
    plot_query_and_candidate_neighbors,
)
from dev.particle_prediction.viz.prediction import (
    plot_local_increment_cloud,
    plot_prediction_fan,
    plot_rollout_against_truth,
    plot_sampled_next_steps,
    plot_support_diagnostics_along_rollout,
)
from dev.particle_prediction.viz.evaluation import (
    plot_error_vs_horizon,
    plot_error_vs_support,
    plot_failure_gallery,
    plot_model_comparison_table,
)

plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)

def resolve_real_data_context(max_default_trajectories=8):
    build_dir = Path(os.environ.get("MORPHSEQ_PARTICLE_BUILD_DIR", project_root / DEFAULT_BUILD_DIR))
    if not build_dir.exists():
        raise FileNotFoundError(
            f"Expected real data under {build_dir}. Set MORPHSEQ_PARTICLE_BUILD_DIR to override."
        )

    available_experiments = sorted(
        path.stem[len(FILE_PREFIX):]
        for path in build_dir.glob(f"{FILE_PREFIX}*.csv")
    )
    if not available_experiments:
        raise FileNotFoundError(f"No {FILE_PREFIX}*.csv files found in {build_dir}")

    env_experiments = os.environ.get("MORPHSEQ_PARTICLE_EXPERIMENTS")
    experiment_ids = [part.strip() for part in env_experiments.split(",") if part.strip()] if env_experiments else [available_experiments[0]]
    env_limit = os.environ.get("MORPHSEQ_PARTICLE_MAX_TRAJECTORIES")
    max_trajectories = int(env_limit) if env_limit else max_default_trajectories
    return build_dir, available_experiments, experiment_ids, max_trajectories

build_dir, available_experiments, experiment_ids, max_trajectories = resolve_real_data_context()
print(f"Project root: {project_root}")
print(f"Build dir: {build_dir}")
print(f"Available experiments: {available_experiments[:5]}{' ...' if len(available_experiments) > 5 else ''}")
print(f"Notebook experiments: {experiment_ids}")
print(f"Trajectory cap: {max_trajectories}")


In [ ]:
dataset = load_trajectories(
    build_dir=build_dir,
    experiment_ids=experiment_ids,
    n_components=10,
    scale=True,
    min_trajectory_length=5,
    verbose=False,
)
selected_trajectories = dataset.trajectories[:max_trajectories]
summary_df = pd.DataFrame(
    {
        "embryo_id": [traj.embryo_id for traj in selected_trajectories],
        "experiment_id": [traj.experiment_id for traj in selected_trajectories],
        "perturbation_class": [traj.perturbation_class for traj in selected_trajectories],
        "n_frames": [len(traj.time_seconds) for traj in selected_trajectories],
        "delta_t": [traj.delta_t for traj in selected_trajectories],
        "n_hard_gaps": [int(np.sum(traj.hard_gap_mask)) if traj.hard_gap_mask is not None else 0 for traj in selected_trajectories],
        "n_interpolatable_gaps": [int(np.sum(traj.interpolatable_gap_mask)) if traj.interpolatable_gap_mask is not None else 0 for traj in selected_trajectories],
    }
)
print(f"Loaded {len(dataset.trajectories)} trajectories; using first {len(selected_trajectories)} for the notebook walkthrough.")
summary_df.head(10)


In [ ]:
window_seconds = 5.0 * 3600.0
poly_order = 2
delta_s = 0.25
history_length = 5
smoothed_subset = smooth_trajectories(selected_trajectories, window_seconds=window_seconds, poly_order=poly_order)
resampled_subset = resample_smoothed_trajectories(smoothed_subset, delta_s=delta_s)
if len(resampled_subset) < 4:
    raise ValueError("Need at least four real trajectories for this evaluation walkthrough.")
split_index = max(2, len(resampled_subset) - 2)
reference_trajectories = resampled_subset[:split_index]
heldout_trajectories = resampled_subset[split_index:]
bank = build_transition_bank(reference_trajectories, history_length=history_length + 2, use_state_index=True)
tasks = build_prediction_tasks(heldout_trajectories, history_length=history_length, horizons=[1, 2, 4], mode="history")
print(f"Reference trajectories: {len(reference_trajectories)} | Held out trajectories: {len(heldout_trajectories)} | Tasks: {len(tasks)}")


## Run the Evaluation Suite on Real Data


In [ ]:
results = run_evaluation_suite(
    tasks=tasks,
    bank=bank,
    n_particles=48,
    random_seed=13,
    matching_config=MatchingConfig(k_state=32, offset_radius=1, retrieval_method="nn"),
)
pd.DataFrame(comparison_table(results)).sort_values("rollout_ade_mean")


## Summary Metrics and Horizon Curves


In [ ]:
plot_model_comparison_table(results)


In [ ]:
plot_error_vs_horizon(results)


## Error Versus Support

These plots help determine whether failure is mostly caused by weak support rather than bad jitter or bad summaries.


In [ ]:
plot_error_vs_support(results, support_key="candidate_count")


In [ ]:
plot_error_vs_support(results, support_key="history_mismatch")


## Failure Cases from Real Trajectories


In [ ]:
plot_failure_gallery(results["local_history"], n_examples=min(6, len(results["local_history"].task_results)))


In [ ]:
pd.DataFrame(
    [
        {
            "model": name,
            **result.support_summaries["candidate_count"],
        }
        for name, result in results.items()
    ]
)


## Real-Data QC Questions

- Do the best metric values correspond to visually sensible rollout fans?
- Are the worst failures concentrated in sparse-support or high-gap regions?
- Does the local-history model actually beat the no-history and fast-summary ablations on the held-out embryos you care about?
